# Bronze Layer — Raw Data Ingestion
**Catalog:** `manufacturing_facility_ops`  
**Schema:** `bronze`  
**Source:** `/Volumes/manufacturing_facility_ops/bronze/raw_data/`  
**Description:** Reads the 5 raw CSV files from the volume and writes them as Delta tables in the bronze schema. Run this notebook whenever source data is refreshed.

In [ ]:
# ── Credentials (managed via AIDP Credential Store) ──────────────
from aidputils.credentials import get_credential

oci_genai_key      = get_credential("oci_genai_api_key")
lakehouse_svc      = get_credential("lakehouse_service_account")
mlflow_token       = get_credential("mlflow_registry_token")

## 0. Setup — Create Schema

In [22]:
spark.sql("CREATE SCHEMA IF NOT EXISTS manufacturing_facility_ops.bronze")

VOLUME_PATH = "/Volumes/manufacturing_facility_ops/bronze/raw_data"
CATALOG     = "manufacturing_facility_ops"
SCHEMA      = "bronze"

print("Schema ready:", f"{CATALOG}.{SCHEMA}")

Schema ready: manufacturing_facility_ops.bronze


## 1. Helper Function — Read CSV and Write Delta

In [23]:
def ingest_csv(filename, table_name):
    path = f"{VOLUME_PATH}/{filename}"
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
    )
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )
    count = spark.table(full_table).count()
    print(f"✅  {full_table}  —  {count} rows loaded")
    return df

## 2. Ingest All 5 Tables

In [21]:
df_machines  = ingest_csv("machines.csv",            "machines")
df_maint     = ingest_csv("maintenance_history.csv", "maintenance_history")
df_orders    = ingest_csv("work_orders.csv",         "work_orders")
df_inventory = ingest_csv("inventory.csv",           "inventory")
df_lines     = ingest_csv("production_lines.csv",    "production_lines")

✅  manufacturing_facility_ops.bronze.machines  —  8 rows loaded


✅  manufacturing_facility_ops.bronze.maintenance_history  —  10 rows loaded


✅  manufacturing_facility_ops.bronze.work_orders  —  10 rows loaded


✅  manufacturing_facility_ops.bronze.inventory  —  10 rows loaded


✅  manufacturing_facility_ops.bronze.production_lines  —  6 rows loaded


## 3. Validation — Row Counts and Column Counts

In [13]:
print(f"{'TABLE':<35} {'ROWS':>6} {'COLS':>6}")
print("-" * 50)
tables = ["machines", "maintenance_history", "work_orders", "inventory", "production_lines"]
for t in tables:
    full  = f"{CATALOG}.{SCHEMA}.{t}"
    df    = spark.table(full)
    print(f"{full:<35} {df.count():>6} {len(df.columns):>6}")

TABLE                                 ROWS   COLS
--------------------------------------------------


manufacturing_facility_ops.bronze.machines      8      8


manufacturing_facility_ops.bronze.maintenance_history     10      9


manufacturing_facility_ops.bronze.work_orders     10     11


manufacturing_facility_ops.bronze.inventory     10      8


manufacturing_facility_ops.bronze.production_lines      6      8


## 4. Preview Tables

In [14]:
print("--- machines ---")
spark.table("manufacturing_facility_ops.bronze.machines").show(8, truncate=False)

--- machines ---


+----------+---------------+---------------+------------+------------+----------------+--------------------+--------+
|machine_id|machine_name   |production_line|status      |install_date|last_maintenance|next_scheduled_maint|location|
+----------+---------------+---------------+------------+------------+----------------+--------------------+--------+
|M-101     |CNC Lathe 101  |Line-1         |Operational |2019-03-12  |2025-04-01      |2025-07-01          |Bay A   |
|M-102     |CNC Lathe 102  |Line-3         |Under Review|2019-03-15  |2024-11-09      |2025-02-09          |Bay A   |
|M-103     |CNC Mill 103   |Line-2         |Operational |2020-06-20  |2025-03-15      |2025-06-15          |Bay B   |
|M-104     |CNC Mill 104   |Line-4         |Operational |2020-06-22  |2025-04-10      |2025-07-10          |Bay B   |
|M-105     |CNC Lathe 105  |Line-5         |Operational |2021-01-08  |2025-04-20      |2025-07-20          |Bay C   |
|M-106     |CNC Lathe 106  |Line-5         |Operational 

In [15]:
print("--- maintenance_history ---")
spark.table("manufacturing_facility_ops.bronze.maintenance_history").show(10, truncate=False)

--- maintenance_history ---


+-------------+----------+------------+-----------------+---------------------------------------------+----------+----------------------------------------+------------+--------+
|work_order_id|machine_id|failure_date|failure_type     |description                                  |technician|resolution                              |downtime_hrs|cost_usd|
+-------------+----------+------------+-----------------+---------------------------------------------+----------+----------------------------------------+------------+--------+
|WO-4310      |M-101     |2024-08-14  |Coolant Leak     |Coolant line fracture at joint B7            |R. Patel  |Replaced coolant line, pressure tested  |4.5         |620     |
|WO-4338      |M-103     |2024-09-02  |Belt Wear        |Primary drive belt showing fraying           |S. Kumar  |Replaced drive belt, aligned pulleys    |3.0         |410     |
|WO-4361      |M-102     |2024-10-17  |Spindle Vibration|Abnormal vibration detected at 3200 RPM      |J. Meht

In [16]:
print("--- work_orders ---")
spark.table("manufacturing_facility_ops.bronze.work_orders").show(10, truncate=False)

--- work_orders ---


+--------+---------------+-------------+----------------+------------------+---------------+----------+----------+--------+-----------+---------------+
|order_id|product        |product_model|quantity_ordered|quantity_completed|production_line|machine_id|due_date  |priority|status     |defect_rate_pct|
+--------+---------------+-------------+----------------+------------------+---------------+----------+----------+--------+-----------+---------------+
|ORD-8801|Washing Machine|WM-7700      |120             |120               |Line-1         |M-101     |2025-05-15|Standard|Completed  |1.2            |
|ORD-8804|Washing Machine|WM-7700      |200             |165               |Line-2         |M-103     |2025-05-17|Standard|In Progress|2.1            |
|ORD-8808|Dryer Unit     |DU-3300      |80              |80                |Line-4         |M-104     |2025-05-16|Standard|Completed  |1.8            |
|ORD-8810|Washing Machine|WM-8800      |150             |62                |Line-3      

In [17]:
print("--- inventory ---")
spark.table("manufacturing_facility_ops.bronze.inventory").show(10, truncate=False)

--- inventory ---


+-----------+---------------------+---------------------+-----------+------------+-------------+-------------+--------------+
|part_number|part_name            |machine_compatibility|warehouse  |bin_location|qty_available|unit_cost_usd|lead_time_days|
+-----------+---------------------+---------------------+-----------+------------+-------------+-------------+--------------+
|SP-4471    |CNC Spindle Assembly |M-101,M-102,M-105    |Warehouse B|B-12-04     |3            |1850         |Same Day      |
|SP-4472    |Drive Belt Primary   |M-103,M-104          |Warehouse A|A-05-11     |8            |120          |Same Day      |
|SP-4490    |Position Encoder Unit|M-104,M-107          |Warehouse A|A-07-03     |2            |530          |Same Day      |
|SP-4501    |Coolant Pump Impeller|M-101,M-108          |Warehouse B|B-08-06     |5            |210          |Same Day      |
|SP-4515    |Tool Holder Assembly |M-103,M-104          |Warehouse A|A-09-02     |4            |310          |Same Day

In [18]:
print("--- production_lines ---")
spark.table("manufacturing_facility_ops.bronze.production_lines").show(6, truncate=False)

--- production_lines ---


+-------+---------------+-----------------------+----------------------+----------------------+----------------------------+-----------+-----------+
|line_id|line_name      |compatible_products    |max_capacity_units_day|current_load_units_day|available_capacity_units_day|shift_hours|status     |
+-------+---------------+-----------------------+----------------------+----------------------+----------------------------+-----------+-----------+
|Line-1 |Assembly Line 1|WM-7700,WM-8800        |180                   |140                   |40                          |16         |Operational|
|Line-2 |Assembly Line 2|WM-7700,CP-110         |200                   |175                   |25                          |16         |Operational|
|Line-3 |Assembly Line 3|WM-8800,WM-9000        |220                   |198                   |22                          |16         |Degraded   |
|Line-4 |Assembly Line 4|DU-3300,DU-4400        |160                   |95                    |65         